In [ ]:
import sys
sys.path.append("..")
import pandas as pd
import sienna
import os
import numpy as np
from utils import detect_column_types 

### Compute statistics

In [18]:
benchmark = "Real Benchmark"
# benchmark = "SINT-Benchmark"
# benchmark = "Join Benchmark"
# benchmark = "Align Benchmark"
# benchmark = "GDS"

In [ ]:
folders = os.listdir(f"data/selected-tables/{benchmark}/")
nr_tables = sum([len([f for f in os.listdir(f"data/selected-tables/{benchmark}/{folder}") if not f.endswith(".json")]) for folder in folders])


In [ ]:
# Data types
data_types_all = {}
data_types_summary = {}
for folder in folders:
    gt_mappings = sienna.load(f"data/gt/{benchmark}/{folder}/{folder}_gt_mappings_with_values.json")
    statistics = {}

    num_numerical = 0
    num_textual = 0
    num_datetime = 0
    total_columns = 0

    other_numerical = 0
    other_textual = 0
    other_datetime = 0

    # Integrated schema by table
    for table_name in gt_mappings["column_mappings"]:
        # Read table
        df = pd.read_csv(f"data/selected-tables/{benchmark}/{folder}/{table_name}")
        total_columns += len(df.columns)

        data_types = detect_column_types(df)

        other_numerical += len(data_types["numerical"])
        other_textual += len(data_types["textual"])
        other_datetime += len(data_types["temporal"])

        data_types_all[table_name] = data_types
    
    data_types_summary[folder] = {
        "numerical": other_numerical,
        "textual": other_textual,
        "datetime": other_datetime,
        "total_columns": total_columns
    }
    print(f"{folder}: Numerical = {other_numerical}, Textual = {other_textual}, Datetime = {other_datetime}, Total = {total_columns}")
    print(f"{folder}: Numerical = {other_numerical/total_columns*100:.2f}%, Textual = {other_textual/total_columns*100:.2f}%, Datetime = {other_datetime/total_columns*100:.2f}%")

In [ ]:
# Percentage of numerical, textual and datetime columns across all tables
print(f"Percentage of numerical columns across all tables: {sum([data_types_summary[folder]['numerical'] for folder in data_types_summary])/sum([data_types_summary[folder]['total_columns'] for folder in data_types_summary])*100:.2f}%")
print(f"Percentage of textual columns across all tables: {sum([data_types_summary[folder]['textual'] for folder in data_types_summary])/sum([data_types_summary[folder]['total_columns'] for folder in data_types_summary])*100:.2f}%")
print(f"Percentage of datetime columns across all tables: {sum([data_types_summary[folder]['datetime'] for folder in data_types_summary])/sum([data_types_summary[folder]['total_columns'] for folder in data_types_summary])*100:.2f}%")

In [ ]:
# Nr of attributes inthe final integrated schema per entity
# folders = ["wikidbs_research-articles", "wikidbs_paintings-collection", "wikidbs_horror-film-character", "wikidbs_monuments", "volcanic-eruptions", "games", "goby_selected", "park_events", "car-listings", "airline-data"]
for folder in folders:
    integrated_gt = sienna.load(f"data/gt/{benchmark}/{folder}/{folder}_gt_integration.json")
    print(f"{folder}: "+', '.join([f"{entity} = {len(integrated_gt['integrated_schema_by_entity'][entity]['attributes'])}" for entity in integrated_gt['integrated_schema_by_entity']]))

In [ ]:
# Nr entities in each table: The average for each use case and calculate the std dev
folders = ["wikidbs_research-articles", "wikidbs_paintings-collection", "wikidbs_horror-film-character", "wikidbs_monuments", "volcanic-eruptions", "games", "goby_selected", "park_events", "car-listings", "airline-data"] #"311_calls_historic_data" "movies",, "500spend",
for folder in folders:
    gt_file = sienna.load(f"data/gt/{folder}/{folder}_gt.json")["schema_by_entity"]
    nr_of_entities = [len(gt_file[table_name]) for table_name in gt_file]
    print(f"{folder}: Average = {np.mean(nr_of_entities):.2f}, Std Dev = {np.std(nr_of_entities):.2f}, Median = {np.median(nr_of_entities):.2f}")

In [ ]:
# Compute max min length, width, density, overlap between attribute tables for each use case
for folder in folders:
    # gt_file = sienna.load(f"../data/gt/{folder}/{folder}_gt.json" )
    gt_mappings = sienna.load(f"data/gt/{benchmark}/{folder}/{folder}_gt_mappings_with_values.json")
    statistics = {}

    # Integrated schema by table
    for table_name in gt_mappings["column_mappings"]:
        # Read table
        df = pd.read_csv(f"data/selected-tables/{benchmark}/{folder}/{table_name}")
        # Drop rows where the first column is NaN
        df = df.dropna(subset=[df.columns[0]])
        column_info = {}
        for column in df.columns:
            column_info[column] = {
                "unique_values": int(df[column].nunique()),
                "missing_values": int(df[column].isna().sum()),
                "missing_percentage": float(df[column].isna().mean() * 100),
                "data_type": str(df[column].dtype)
            }
        # Add to statistics file the length, width and density of each column and of all the table
        statistics[table_name] = {
            "num_rows": len(df),
            "num_columns": len(df.columns),
            "table_missing_percentage": float(sum([column_info[column]["missing_percentage"] for column in df.columns])/len(df.columns)) if len(df.columns) > 0 else 0,
            "column_info": column_info
        }
    
    # Calculate min, max, average length, width, missing percentage of tables and standard deviation
    lengths = [statistics[table_name]["num_rows"] for table_name in statistics]
    widths = [statistics[table_name]["num_columns"] for table_name in statistics]
    missing_percentages = [statistics[table_name]["table_missing_percentage"] for table_name in statistics]
    statistics["overall"] = {}
    statistics["overall"]["length"] = {
        "min": min(lengths) if lengths else 0,
        "max": max(lengths) if lengths else 0,
        "average": sum(lengths)/len(lengths) if lengths else 0,
        "median": sorted(lengths)[len(lengths)//2] if lengths else 0,
        "std": np.std(lengths) if lengths else 0
    }
    statistics["overall"]["width"] = {
        "min": min(widths) if widths else 0,
        "max": max(widths) if widths else 0,
        "average": sum(widths)/len(widths) if widths else 0,
        "median": sorted(widths)[len(widths)//2] if widths else 0,
        "std": np.std(widths) if widths else 0
    }
    statistics["overall"]["missing_percentage"] = {
        "min": min(missing_percentages) if missing_percentages else 0,
        "max": max(missing_percentages) if missing_percentages else 0,
        "average": sum(missing_percentages)/len(missing_percentages) if missing_percentages else 0,
        "median": sorted(missing_percentages)[len(missing_percentages)//2] if missing_percentages else 0
    }

    # Calculate overlap between attribute tables
    column_mappings = gt_mappings["column_mappings"]
    # For each right-side calculate how many overlap it has with other right-sides
    overlap_counts = {}
    for table_name in column_mappings:
        overlap_counts[table_name] = {}

        for other_table_name in column_mappings:
            if other_table_name != table_name:
                # Calculate the overlap between the two sets of right-sides
                other_attributes = column_mappings[other_table_name].values()
                attributes = column_mappings[table_name].values()
                nr_overlap = len(set(attributes).intersection(set(other_attributes)))
                overlap_counts[table_name][other_table_name] = nr_overlap/len(set(attributes).union(set(other_attributes)))

    
    # Add to statistics file the overlap counts
    for table_name in overlap_counts:
        statistics[table_name]["overlap_count"] = sum(overlap_counts[table_name].values())
        statistics[table_name]["overlap_percentage"] = float(sum(overlap_counts[table_name].values())/len(overlap_counts[table_name].values()))

    statistics["overall"]["overlap_percentage"] = float(sum([statistics[table_name]["overlap_percentage"] for table_name in statistics if table_name != "overall"]))/len([table_name for table_name in statistics if table_name != "overall"]) if len([table_name for table_name in statistics if table_name != "overall"]) > 0 else 0
    # Calculate std of overlap percentage
    statistics["overall"]["overlap_percentage_std"] = np.std([statistics[table_name]["overlap_percentage"] for table_name in statistics if table_name != "overall"]) if len([table_name for table_name in statistics if table_name != "overall"]) > 0 else 0

    # Save statistics file
    sienna.save(statistics, f"data/statistics/{benchmark}_{folder}__statistics.json")

In [ ]:
# Create a df with main statistics from each folder
all_stats = []
for folder in folders:
    statistics = sienna.load(f"data/statistics/{benchmark}_{folder}__statistics.json")
    # Create a df with the overall statistics of each folder
    overall_stats = statistics["overall"]
    # Unwrap dictionary to have columns like overall_length_min, overall_length_max, etc.
    row_stats = {}
    row_stats["folder"] = folder
    for key, subdict in overall_stats.items():
        if isinstance(subdict, dict):
            # print(f"Unwrapping {key}")
            for subkey, value in subdict.items():
                # print(f"overall_{key}_{subkey}")
                # print(value)

                row_stats[f"overall_{key}_{subkey}"] = value
        else:
            row_stats[f"overall_{key}"] = subdict
    all_stats.append(row_stats)

In [ ]:
overall_statistics_df = pd.DataFrame(all_stats)

In [ ]:
overall_statistics_df.to_csv(f"../data/statistics/{benchmark}_all_use_cases_statistics.csv", index=False)

In [ ]:
# overall_statistics_df = pd.read_csv(f"../data/statistics/{benchmark}_all_use_cases_statistics.csv")
overall_statistics_df

In [ ]:
# Average of all
print(f"Average length: {overall_statistics_df['overall_length_average'].mean():.2f}")
print(f"Average length std: {overall_statistics_df['overall_length_std'].mean():.2f}")
print(f"Average width: {overall_statistics_df['overall_width_average'].mean():.2f}")
print(f"Average width std: {overall_statistics_df['overall_width_std'].mean():.2f}")
print(f"Average missing percentage: {overall_statistics_df['overall_missing_percentage_average'].mean():.2f}%")
print(f"Average overlap percentage: {overall_statistics_df['overall_overlap_percentage'].mean():.2f}")
print(f"Average overlap percentage std: {overall_statistics_df['overall_overlap_percentage_std'].mean():.2f}")